# MQM 0.5 · subsystem [[8,1,3]], three gauge qubits
18 September 2026. Exact Stim Pauli algebra, 65,536 phase-quotiented labels, a weight-three witness, and a newly specified ideal-syndrome decoder. This is not the missing noisy memory sweep or its flag decoder. No hardware result; no claimed advantage.
Run the code below after installing requirements.txt. Pauli phases are ignored only for group membership/distance enumeration. The full gauge group is noncommuting; its generators must not be treated as simultaneous stabilizers.


In [1]:
import json
from itertools import product
import stim

CENTER = ['IIIXXZZI', 'IIIYIYZZ', 'IIIXZIXZ', 'XXXXYYIX']
GAUGE = CENTER + ['XIIIIIII','ZIIIIIIZ','IXIIIIII','IZIIIIIZ','IIXIIIII','IIZIIIIZ']
LX, LZ = 'IIIYZIZI', 'IIIXIZIZ'

def key(p):
    return ''.join('IXYZ'[p[i]] for i in range(len(p)))

def span(labels):
    group={'I'*8}
    for label in labels:
        g=stim.PauliString(label)
        group |= {key(stim.PauliString(p)*g) for p in tuple(group)}
    return group

group, center = span(GAUGE), span(CENTER)
gs, cs = [stim.PauliString(p) for p in GAUGE], [stim.PauliString(p) for p in CENTER]
assert len(group)==1024 and len(center)==16
assert {p for p in group if all(stim.PauliString(p).commutes(g) for g in gs)}==center
x,z=stim.PauliString(LX),stim.PauliString(LZ)
assert all(l.commutes(g) for l in (x,z) for g in gs)
assert not x.commutes(z) and key(x) not in group and key(z) not in group
counts={}; witness=None; distance=9
# A new ideal-syndrome reference decoder. Not the missing historical flag decoder.
decoder={}
for labels in product('IXYZ',repeat=8):
    label=''.join(labels);p=stim.PauliString(label);weight=p.weight
    syndrome=''.join(str(int(not p.commutes(s))) for s in cs)
    if syndrome not in decoder or (weight,label)<(stim.PauliString(decoder[syndrome]).weight,decoder[syndrome]):
        decoder[syndrome]=label
    if syndrome=='0000' and label not in group:
        counts[str(weight)]=counts.get(str(weight),0)+1
        if weight<distance: distance=weight;witness=label
assert distance==3 and witness=='IIIIIXXZ'
for a,b in [(0,1),(0,2),(0,7),(1,2),(1,7),(2,7)]:
    p=['I']*8;p[a]=p[b]='Z';assert ''.join(p) in group
# Verify the ideal reference decoder on every single-data Pauli error.
for q in range(8):
    for c in 'XYZ':
        label=['I']*8;label[q]=c;p=stim.PauliString(''.join(label))
        syndrome=''.join(str(int(not p.commutes(s))) for s in cs)
        assert key(p*stim.PauliString(decoder[syndrome])) in group
result={'release':'0.5','stim_version':stim.__version__,'n_data':8,'n_logical':1,'n_gauge':3,
 'gauge_binary_rank':10,'center_binary_rank':4,'dressed_distance':distance,'witness':witness,
 'nontrivial_dressed_logical_counts_by_weight':counts,'single_data_Pauli_decoder_cases':24,
 'ideal_reference_decoder':decoder,
 'scope':'Phase-quotiented exact algebra and ideal syndrome lookup only; no noisy extraction or memory experiment.'}
print(json.dumps(result,indent=2))


{
  "release": "0.5",
  "stim_version": "1.15.0",
  "n_data": 8,
  "n_logical": 1,
  "n_gauge": 3,
  "gauge_binary_rank": 10,
  "center_binary_rank": 4,
  "dressed_distance": 3,
  "witness": "IIIIIXXZ",
  "nontrivial_dressed_logical_counts_by_weight": {
    "3": 66,
    "5": 804,
    "4": 234,
    "6": 900,
    "7": 666,
    "8": 402
  },
  "single_data_Pauli_decoder_cases": 24,
  "ideal_reference_decoder": {
    "0000": "IIIIIIII",
    "0110": "IIIIIIIX",
    "0111": "IIIIIIIY",
    "0001": "IIIIIIIZ",
    "1100": "IIIIIIXI",
    "1010": "IIIIYIII",
    "1011": "IIIYIIII",
    "1101": "IIIIIXII",
    "1110": "IIIIIIYI",
    "1000": "IIIIIYII",
    "1001": "IIIIZIII",
    "1111": "IIIZIIII",
    "0010": "IIIIIIZI",
    "0100": "IIIXIIII",
    "0101": "IIIIIZII",
    "0011": "IIIIXIII"
  },
  "scope": "Phase-quotiented exact algebra and ideal syndrome lookup only; no noisy extraction or memory experiment."
}
